In [ ]:
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Detector Error Models and Real-Time Decoding — QEC 101
$\renewcommand{\ket}[1]{|#1\rangle}$

---

So far the QEC 101 labs have focused primarily on QEC memory — the specific construction of logical qubit encodings with techniques like the repetition code, Steane and Shor codes, the surface code, and qLDPC codes. Though memory is foundational to QEC, it can only correct errors from a simple noise model that ignores multiple sources of noise when an algorithm or QEC routine is run.

Similarly, we have not explored deployment of any codes in a real-time QEC workflow. This includes consideration of decoder metrics like throughput and reaction time covered in "[Decoder Metrics and (Temporal) Parallel Window Decdoing](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/08_QEC_Decoder_metrics_and_parallel_window_decoding.ipynb)" but also specific implementation details like what APIs are necessary to perform QEC, keeping track of the correct data, and performing computations on the right processor.

This lab will explore the limitations of memory experiments and motivate the need for a more robust tool for decoding how errors arise in practice during QEC rounds. The detector error model (DEM), introduced in a paper by Eisert and coworkers called ["Designing fault-tolerant circuits using detector error models"](https://quantum-journal.org/papers/q-2025-11-06-1905/pdf/), will be presented as a solution to this problem. You will explore some of the theory behind DEMs and some of the practical aspects of deploying them for real-time decoding.

**What You Will Do:**
* Explore the limitations of QEC memory and why code capacity assumptions are insufficient for real-time decoding
* Construct detector error models (DEMs) for the repetition code under different noise models
* Compute detector error matrices and verify error identification using syndromes
* Analyze observables and undetectable error patterns using Tanner graphs
* Initialize a real-time decoder in CUDA-Q QEC using a DEM for the Steane code

**Prerequisites:**
* Python and Jupyter familiarity
* Completion of QEC 101 labs 1–4, especially [Stabilizers](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/02_QEC_Stabilizers.ipynb) and [Decoders](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/04_QEC_Decoders.ipynb)
* Familiarity with the Steane code and repetition code encodings
* Basic understanding of noise models in quantum computing

**Key Terminology:**
* Detector error model (DEM)
* Detector
* Detector matrix
* Measurement syndrome matrix
* Phenomenological noise model
* Code capacity
* Circuit-level noise model

**CUDA-Q Syntax:**
* [`@cudaq.kernel`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.kernel) — defines a quantum kernel function
* [`cudaq.qvector`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.qvector) — allocates a register of qubits
* [`cudaq.sample`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.sample) — samples measurement outcomes from a kernel
* [`cudaq.set_target`](https://nvidia.github.io/cuda-quantum/latest/api/languages/python_api.html#cudaq.set_target) — selects simulation or hardware backend
* [`cudaq_qec.patch`](https://nvidia.github.io/cuda-quantum/latest/api/qec/python_api.html) — logical qubit register with data and ancilla sub-registers
* [`cudaq_qec.get_code`](https://nvidia.github.io/cuda-quantum/latest/api/qec/python_api.html) — loads a pre-built QEC code definition
* [`cudaq_qec.z_dem_from_memory_circuit`](https://nvidia.github.io/cuda-quantum/latest/api/qec/python_api.html) — builds a detector error model from a memory circuit
* [`cudaq_qec.decoder_config`](https://nvidia.github.io/cuda-quantum/latest/api/qec/python_api.html) — configures decoder parameters
* [`cudaq_qec.enqueue_syndromes`](https://nvidia.github.io/cuda-quantum/latest/api/qec/python_api.html) — sends syndrome data to a decoder
* [`cudaq_qec.get_corrections`](https://nvidia.github.io/cuda-quantum/latest/api/qec/python_api.html) — retrieves logical corrections from a decoder
* [`cudaq_qec.configure_decoders_from_file`](https://nvidia.github.io/cuda-quantum/latest/api/qec/python_api.html) — loads decoder configuration from YAML

**Solutions:** [`Solutions/09_QEC_Detector_Error_Models_Solution.ipynb`](Solutions/09_QEC_Detector_Error_Models_Solution.ipynb)

In [ ]:
## Instructions for Google Colab. You can ignore this cell if you have CUDA-Q
## set up locally with all required files on your system.
## Uncomment the lines below and execute this cell to install CUDA-Q.

#!pip install cudaq -q
#
#!wget -q https://github.com/nvidia/cuda-q-academic/archive/refs/heads/main.zip
#!unzip -q main.zip
#!mv cuda-q-academic-main/qec101/Images ./Images

> **Note:** Run the cell below to import all required packages.
> If you installed packages above, restart the kernel first
> (**Runtime → Restart session** in Colab, or **Kernel → Restart** in Jupyter).

In [1]:
import os

import numpy as np

import cudaq

## To install cudaq-qec (if not already installed), uncomment and run:
## !pip install cudaq-qec -q
import cudaq_qec as qec

---

## 1. Real-Time Decoding and the Limits of Code Capacity

Real-time decoding is an extremely difficult problem that requires a number of key ingredients. First, a QEC code is needed that efficiently encodes logical qubits. There are lots of considerations here, including qubit topology, qubit count, code distance, etc. Recall that in general, the goal is a logical qubit encoding scheme that can capture as many errors as possible using as few physical qubits as possible.

Next, the logical qubits need to perform the logic necessary to run fault-tolerant algorithms and ensure that QEC cycles can continuously run to catch errors. These cycles involve measuring syndromes from the QPU, decoding them in the decoder, and sending the identified errors back to the QPU as shown in the figure below.

<figure>
  <img src="Images/dem/qec_workflow.png" alt="Diagram of a QEC workflow loop showing syndrome extraction on a QPU, transmission of syndrome data to a classical decoder, and return of error corrections back to the QPU" width="600">
</figure>

The notebook on ["Decoder Metrics and Parallel Window Decoding"](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/08_QEC_Decoder_metrics_and_parallel_window_decoding.ipynb) explored the specific constraints that arise here in detail. For example, the throughput of the decoder must be greater than the rate at which syndromes are generated from the QPU. If not, the process grinds to a halt. Likewise, the reaction time, or how quickly the decoder can return results to the QPU, will determine QPU wall clock speed.

A problem not yet discussed in a practical setting is what actually goes into preparation of the decoder. In previous notebooks, the parity check matrix that defined the QEC code memory was always the input. This is perfectly fine if we prepare say $\ket{0}_L$ with the Steane code, allow an error to occur, and *then* run a stabilizer round to check for the error (i.e., **code capacity** assumptions). The issue is that assuming errors only occur on the data qubits between state prep and the stabilizer rounds is quite unrealistic. In a physical QPU, errors can occur on any qubit at any time!

Explore the [Steane Code Error Models widget](https://nvidia.github.io/cuda-q-academic/interactive_widgets/steane_code_error_models.html) to test this with the Steane code. Confirm an error before the first stabilizer begins can be correctly identified. Next, see what happens when an error occurs before an ancilla measurement (choose circuit-level noise model). Finally, see what happens when an error occurs between the stabilizer extractions (choose phenomenological noise model).

It should now be clear that quantum memory is not enough and any practical QEC routine needs to feed the decoder something more robust than the parity check matrix for encoding the logical qubits.

Part of the solution has already been explored in the notebook on [decoders](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/04_QEC_Decoders.ipynb). Recall that decoders do not just work in space but also decode in time, taking multiple syndrome extraction circuits at once. Consider the Steane code again. Assume that only a single measurement error occurs on the first of five syndrome extraction cycles. If only the first round was decoded in isolation, it would be incorrectly assumed an error occurred. If all five are examined at once, it would be instead clear that a one-time measurement error occurred.

This is just one example demonstrating why the decoder needs to be provided with an object more robust than a parity check matrix that can account for errors occurring anywhere in the circuit and the results of multiple stabilizer measurement rounds.

---

## 2. Detector Error Models (DEMs)

**Detector error models (DEMs)** are powerful constructs that solve the problem described above by constructing a more robust detector error matrix that can initialize a decoder to flag errors at any possible location. To fully appreciate what a DEM is and why it works, it is worth walking through some of the theory. DEMs were first described in ["Designing fault-tolerant circuits using detector error models"](https://quantum-journal.org/papers/q-2025-11-06-1905/pdf/) and the next few sections of this lab will cover some of the the key definitions and let you try a number of exercises drawn from the paper.

The first piece we need is to define a **detector** $d_i$. A detector is a sum of measurements corresponding to a parity constraint that arises from a specific circuit. For example, $m_1 \oplus m_2 \oplus m_3 =0$. The detector can be represented by a vector of length $m$ where $m$ is the number of measurements in a circuit and 1's indicate measurements included in the sum.

$$d_1 =\begin{bmatrix}
1 \\
1 \\
1
\end{bmatrix}$$

Stated otherwise, if $d_1$ is a valid detector for some circuit, it will always be true (absent of noise) that all three measurements result in a binary sum of $b_i$, where $b_i$ is the expected binary sum (Usually 0 for the examples in this lesson).

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 1:</span>**

Code the circuit below in CUDA-Q. Prove (by sampling or by stabilizer tracking) that $d_1$ is a detector for this circuit with an expected $b_i$=0.


<img src="Images/dem/exercise1_circuit.png" alt="Three-qubit circuit with initial states |0⟩, |+⟩, and |−⟩, two CNOT gates, and three measurements M1, M2, M3" width="500">

</div>

In [2]:
# EXERCISE 1

@cudaq.kernel
def example1():
    reg = cudaq.qvector(3)

    h(reg[1])
    x(reg[2])
    h(reg[2])

    x.ctrl(reg[1], reg[0])
    x.ctrl(reg[2], reg[1])

print(cudaq.sample(example1))

{ 000:231 011:263 101:248 110:258 }



The only way for this not to hold true for this circuit would be if an error occurred. We say that a detector is violated if
$d_i^Tm\neq b_i$, where $m$ is a vector of measurement outcomes.

More complex circuits can have multiple detectors which can combine to form a **detector matrix** where each row of the matrix is a different detector defined in the absence of noise. These detectors must be linearly independent otherwise they are providing redundant constraints.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 2:</span>**

Consider the repetition code circuit below. Define a detector matrix which has four detectors. Use this detector matrix to prove that a measurement error would violate at least one detector. Hint: assume all qubits begin in the 0 state.

</div>

<figure>
  <img src="Images/dem/dem_rep.png" alt="Circuit diagram of a three-qubit repetition code showing data qubit initialization, CNOT encoding gates, and two ancilla-based stabilizer measurement rounds with five total measurements labeled m1 through m5" width="600">
</figure>

In [4]:
# EXERCISE 2

D = np.array([[1, 0, 0, 0, 0],
              [0, 1, 0 ,0, 0], 
              [0, 0, 1, 1, 0],
              [0, 0, 0, 1, 1]])


print("Single measurement error analysis:")
for i in range(D.shape[1]):
    m = np.zeros(D.shape[1], dtype=int)
    m[i] = 1 # apply measurement error on qubit i
    violated = (D @ m) % 2
    print(f"  Error on m_{i+1}: violated detectors = {np.where(violated)[0] + 1}")

Single measurement error analysis:
  Error on m_1: violated detectors = [1]
  Error on m_2: violated detectors = [2]
  Error on m_3: violated detectors = [3]
  Error on m_4: violated detectors = [3 4]
  Error on m_5: violated detectors = [4]


With detectors in hand, we can begin to expand our noise model and build out a few more pieces of the DEM. Consider the same circuit now with errors added before the stabilizers and before any measurement. This is a **circuit-level noise model** where "any n-qubit Pauli error can occur after an n-qubit gate and any single Pauli error after state preparation."

<figure>
  <img src="Images/dem/dem_rep_clevel.png" alt="Circuit diagram of the three-qubit repetition code with circuit-level noise, showing error locations labeled E1 through E8 inserted after each gate and before each measurement" width="600">
</figure>

With this, or any other noise model, we can define a circuit error vector $e$ which has a 1 if error $E_i$ occurred and 0 if not. This allows us to construct the **measurement syndrome matrix** $\Omega$ which maps errors to measurements. Each row corresponds to the measurements in the circuit, 5 in this case. Each column corresponds to one of the 8 possible error locations. 1's are populated if error $j$ would flip measurement $i$.

In practice, $\Omega$ needs to be computed via Pauli propagation for large and more complex circuits. Once obtained, we can compute a **detector error matrix** $H = D\Omega$. $H$ is similar to what we previously called a parity check matrix, but now contains circuit-level information in addition to QEC memory information. The structure of $H$ is such that the rows correspond to detectors and the columns to errors. A 1 entry means error $j$ would violate detector $i$.

Similar to a standard parity check matrix, we now call $s$ a syndrome obtained from $s = He$. This means that we can now identify errors based on which detectors are flagged, capturing information about QEC memory and the circuit-level noise!

Similarly, DEMs can be extended to encompass an even broader **phenomenological noise model** where errors can happen between syndrome extraction gates. We will not cover this here, but the authors present so-called measurement schedules which can cleverly design syndrome extraction circuits such that a given routine is fault-tolerant by design and error will not propogate uncontrollably.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 3:</span>**

Complete $\Omega$ for the circuit-level noise model of the repetition code (image above with the 8 error locations). Use $\Omega$ to compute $H$. For every weight-1 error, compute the syndrome and prove that each error corresponds to a unique syndrome. Note that we can capture more types of errors using a DEM, but are still constrained by the code distance ($d=3$) for how many errors can be flagged.

</div>

In [5]:
# EXERCISE 3

omega = np.array([[1, 1, 0, 1, 0, 0, 0, 0],
                  [0, 1, 1 ,0, 1, 0, 0 ,0], 
                  [1, 0, 0, 0, 0, 1, 0, 0],
                  [0, 1, 0, 0, 0, 0, 1, 0],
                  [0, 0, 1, 0, 0, 0, 0, 1]]) 

H = D @ omega

print(H)


print("Weight 1 Error Syndromes")
for i in range(8):
    error = np.zeros(8)
    error[i] = 1
    print("Error Location i =", i)
    print(H @ error)

[[1 1 0 1 0 0 0 0]
 [0 1 1 0 1 0 0 0]
 [1 1 0 0 0 1 1 0]
 [0 1 1 0 0 0 1 1]]
Weight 1 Error Syndromes
Error Location i = 0
[1. 0. 1. 0.]
Error Location i = 1
[1. 1. 1. 1.]
Error Location i = 2
[0. 1. 0. 1.]
Error Location i = 3
[1. 0. 0. 0.]
Error Location i = 4
[0. 1. 0. 0.]
Error Location i = 5
[0. 0. 1. 0.]
Error Location i = 6
[0. 0. 1. 1.]
Error Location i = 7
[0. 0. 0. 1.]


---

## 3. Observables and DEMs

We can utilize the DEM model to explore computation of observables where an observable $o_i$ is "a binary sum of measurements, which equals the outcome of measuring a logical operator, for any logical state." For example, the figure below is a repetition code with two QEC cycles. A valid choice for a logical $Z$ observable is $o = m_5$. Without noise, a single measurement of any of the data qubits determines if the state is 0 if $\ket{0_L}$ or 1 if $\ket{1_L}$.

<figure>
  <img src="Images/dem/dem_two_rep.png" alt="Circuit diagram of a three-qubit repetition code with two rounds of syndrome extraction, showing seven total measurements m1 through m7 and data qubit measurements at the end" width="600">
</figure>

Other choices are valid, for example, $m_5 \oplus m_6 \oplus m_7$ would also work. The key is to select a constraint that depends on the logical state where the other detectors do not depend on the logical state.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 4:</span>**

Use the following detectors to build $D$ for the two-round repetition code circuit. Also code the circuit in CUDA-Q. Using the Tanner graph below, try to identify an error pattern that flips the observable and does not trip any detectors. How many errors are required for this? Test one of these cases using your CUDA-Q code by manually entering bitflips, sampling the circuit, and confirming the resulting bitstring(s) do not flag any detectors.

$$d_1: m_1 = 0$$
$$d_2: m_2 = 0$$
$$d_3: m_1 \oplus m_3 = 0$$
$$d_4: m_2 \oplus m_4 = 0$$
$$d_5: m_3 \oplus m_5 \oplus m_6 = 0$$
$$d_6: m_4 \oplus m_6 \oplus m_7 = 0$$

</div>

<figure>
  <img src="Images/dem/tanner.png" alt="Tanner graph showing connections between error nodes and detector nodes for the two-round repetition code, with the logical observable node highlighted" width="400">
</figure>

In [ ]:
# EXERCISE 4

D = np.array([[1, 0, 0, 0, 0, 0, 0],
              [0, 1, 0 ,0, 0, 0, 0], 
              [1, 0, 1, 0, 0, 0, 0],
              [0, 1, 0, 1, 0, 0, 0],
              [0, 0, 1, 0, 1, 1, 0],
              [0, 0, 0, 1, 0, 1, 1]]) 


# Example with E_7 E_8 E_9 E_11

@cudaq.kernel
def exercise4():
    ancilla = cudaq.qvector(4)
    reg = cudaq.qvector(3)

    #state prep 0_L
    x.ctrl(reg[0], reg[1])
    x.ctrl(reg[0], reg[2])
    
    #M1
    x.ctrl(reg[0], ancilla[0])
    x.ctrl(reg[1], ancilla[0])

    #M2
    x.ctrl(reg[1], ancilla[1])
    x.ctrl(reg[2], ancilla[1])

    x(reg[1])
    x(reg[2])

    #M3
    x.ctrl(reg[0], ancilla[2])
    x.ctrl(reg[1], ancilla[2])

    #M4
    x.ctrl(reg[1], ancilla[3])
    x.ctrl(reg[2], ancilla[3])

    x(reg[0]) # E9
    x(ancilla[2]) #E11
    
sample = cudaq.sample(exercise4)
print(sample)

for bitstring in sample:
    bitstring_np = np.fromiter((int(b) for b in bitstring), dtype=int)

print((D @ bitstring_np)%2)


{ 0000111:1000 }

[0 0 0 0 0 0]


---

## 4. Initializing a Decoder in CUDA-Q with a DEM for Real-Time Decoding

Aside from when the decoding occurs, what differentiates real-time decoding from how we think about offline decoding?

The answer revolves around how the workflow is implemented in practice and where specific computations occur. Recall the workflow below. Though it is correct in general, it is oversimplified and hides some of the nuance.

<figure>
  <img src="Images/dem/qec_workflow.png" alt="Diagram of a QEC workflow loop showing syndrome extraction on a QPU, transmission of syndrome data to a classical decoder, and return of error corrections back to the QPU" width="700">
</figure>

For example, the decoder actually decodes syndromes that are XOR'd with the previous syndrome in order to flag differences (often called detector events). Where does this XOR calculation occur? It cannot be the QPU which only outputs raw syndrome measurements. Similarly, the decoding process might output inferred errors, but all that matters is whether a logical flip occurred. At some point this needs to be determined from the actual decoding solution.

A more slightly more realistic (but still simplified) real-time workflow is the following where the QPU only produces measurements and receives determination if logical flips occurred or not. The decoder must then handle everything else.

<figure>
  <img src="Images/dem/rt_qec_workflow.png" alt="Diagram of a real-time QEC workflow showing the QPU producing raw measurements, a classical processing layer performing XOR differencing and syndrome extraction, a decoder returning logical flip determinations, and the classical control system relaying corrections to the QPU" width="700">
</figure>

This means when writing code for quantum algorithms, we need APIs that can be called within a quantum kernel to send data to a decoder and retrieve results when finished. CUDA-Q QEC enables real-time decoding workflows that look like the following.

The first key function is `qec.enqueue_syndromes` which takes measurement data from the QPU and sends it to a preconfigured decoder. The second is `qec.get_corrections` which simply returns the logical flips so they can be applied and the algorithm can proceed. Everything else is handled within a preinitialized decoder. Such a construct is important because a more complex quantum algorithm may be sending data to multiple decoders depending on the scenario. Each can be initialized and prepared to use any technique necessary for the job.

So, what does this have to do with DEMs? A real-time decoding workflow involves preparation of a decoder using a DEM. This results in a robust decoder that can handle a range of error models and process syndromes from the QPU with a single function call.

<div style="background-color: #f9fff0; border-left: 6px solid #76b900; padding: 15px 15px 15px 20px; border-radius: 4px; margin: 15px 0; color: #333;">

**<span style="color: #76b900; font-size: 1.17em;">Exercise 5:</span>**

Work through and run the code cells below to initialize a decoder with a DEM in CUDA-Q QEC for the Steane code. Along the way, you might be asked to fix the code or enter values as prompted.

</div>

First, we need to define a few helper functions that prepare the logical zero state and measure the stabilizer checks for the Steane code. We need to define two kernels, one to prepare the Steane code $\ket{0_L}$ state and another to define measurement of the $Z$ stabilizers to check for $X$ errors. (We are ignoring $Z$ errors in this example for simplicity.) Both kernels should take a `qec.patch` object as an input and act on its `logical.data[i]`, `logical.ancz[i]`, or `logical.ancx[i]` registers. You can directly import the Steane code from CUDA-Q QEC, but you can also define it manually which is helpful if you bring your own code.

Both of these kernels take a `qec.patch` object as input which is similar to providing a register, but contains three registers for $X$ and $Z$ check ancillas and the data qubits.

In [ ]:
# EXERCISE 5

os.environ["CUDAQ_DEFAULT_SIMULATOR"] = "stim"

# Prepare logical |0⟩
@cudaq.kernel
def prep0(logical: qec.patch):
    h(logical.data[4])
    h(logical.data[5])
    h(logical.data[6])

    x.ctrl(logical.data[0],logical.data[1])
    x.ctrl(logical.data[0],logical.data[2])

    x.ctrl(logical.data[4],logical.data[0])
    x.ctrl(logical.data[4],logical.data[1])
    x.ctrl(logical.data[4],logical.data[3])

    x.ctrl(logical.data[5],logical.data[0])
    x.ctrl(logical.data[5],logical.data[2])
    x.ctrl(logical.data[5],logical.data[3])

    x.ctrl(logical.data[6],logical.data[1])
    x.ctrl(logical.data[6],logical.data[2])
    x.ctrl(logical.data[6],logical.data[3])


# Measure Z stabilizers for Steane code
@cudaq.kernel
def measure_stabilizers_z(logical: qec.patch) -> list[bool]:

    for i in range(logical.ancz.size()):
        reset(logical.ancz[i])

    h(logical.ancz)

    z.ctrl(logical.ancz[0],logical.data[0])
    z.ctrl(logical.ancz[0],logical.data[1])
    z.ctrl(logical.ancz[0],logical.data[3])
    z.ctrl(logical.ancz[0],logical.data[4])

    z.ctrl(logical.ancz[1],logical.data[0])
    z.ctrl(logical.ancz[1],logical.data[2])
    z.ctrl(logical.ancz[1],logical.data[3])
    z.ctrl(logical.ancz[1],logical.data[5])

    z.ctrl(logical.ancz[2],logical.data[1])
    z.ctrl(logical.ancz[2],logical.data[2])
    z.ctrl(logical.ancz[2],logical.data[3])
    z.ctrl(logical.ancz[2],logical.data[6])

    h(logical.ancz)

    return [mz(logical.ancz[0]), mz(logical.ancz[1]), mz(logical.ancz[2])]

Next, the main QEC circuit is prepared. This prepares a set of registers for the data and ancilla qubits called a patch stored as the variable `logical`. Enter the correct number of qubits for each register.

After preparing the initial state, three syndrome extraction cycles are run. The syndromes are simply obtained from the measurement outcomes of the stabilizer circuit you defined above. These are fed into the function `enqueue_syndromes` which takes the decoder ID, the syndromes, and an optional flag for debugging. Note that in complex QEC workflows, it is likely that multiple decoders will be used, hence a command is needed to send syndromes to any specified decoder.

The decoder then follows any instructions it has and completes decoding.

Finally, the corrections are obtained from the same decoder ID with `get_corrections` and the number of logical observables (3 in the case of three rounds). The corrections are then applied to the data qubits before measurement.

In [ ]:
@cudaq.kernel
def qec_circuit() -> list[bool]:
    qec.reset_decoder(0)

    data = cudaq.qvector(7)
    ancz = cudaq.qvector(3)
    ancx = cudaq.qvector(0) # Keep 0 as we are ignoring Z errors
    logical = patch(data, ancx, ancz)

    prep0(logical)

    # 3 rounds of syndrome measurement
    for _ in range(3):
        syndromes = measure_stabilizers_z(logical)
        qec.enqueue_syndromes(0, syndromes, 0)

    # Get corrections and apply them
    corrections = qec.get_corrections(0, 3, False)
    for i in range(3):
        if corrections[i]:
            x(data[i])

    return mz(data)

Now that the prerequisites are complete, we can begin the main workflow and decoder initialization. First, use `get_code` to load information about the Steane code from the pre-built code information.

Next, define a noise model. In the cell below, a depolarization error model is applied to all CNOT gates with a probability of 0.01.

Then, call `z_dem_from_memory_circuit` to build a DEM given the QEC code, the logical state prep circuit, the number of rounds, and the noise model. Under the hood, this calculates the syndrome measurement matrix ($\Omega$) and selects a set of detectors. The set of detectors picked by this function is valid, but may not be optimal.

In [ ]:
code = qec.get_code("steane", distance=3)

# [Begin DEM Generation]
print("Step 1: Generating DEM...")
cudaq.set_target("stim")

noise = cudaq.NoiseModel()
noise.add_all_qubit_channel("x", cudaq.Depolarization2(0.01), 1)

dem = qec.z_dem_from_memory_circuit(code, qec.operation.prep0, 3, noise)

print(dem.detector_error_matrix)

The decoder can be set up by setting a number of configurations and saving them in a YAML file. Notice these settings involve selecting the decoder type, how large the decoding problem is, sparse representation of the DEM, etc. The main idea is that you can have lots of flexibility when defining a decoder to balance the many different tradeoffs and that many of the settings come directly from the structure of the DEM. The [CUDA-Q QEC docs](https://nvidia.github.io/cudaqx/components/qec/introduction.html) provide details on all of the different settings.

In [ ]:
config = qec.decoder_config()   
config.id = 0 # Sets decoder ID
config.type = "multi_error_lut" # Specifies decoding algorithm
config.block_size = dem.detector_error_matrix.shape[1]
config.syndrome_size = dem.detector_error_matrix.shape[0]
config.H_sparse = qec.pcm_to_sparse_vec(dem.detector_error_matrix)
config.O_sparse = qec.pcm_to_sparse_vec(dem.observables_flips_matrix)

# Calculate numRounds from DEM (we send 1 additional round, so add 1)
num_syndromes_per_round = 3  
num_rounds = dem.detector_error_matrix.shape[0] // num_syndromes_per_round + 1
config.D_sparse = qec.generate_timelike_sparse_detector_matrix(num_syndromes_per_round, num_rounds, False)
lut_config = qec.multi_error_lut_config()
lut_config.lut_error_depth = 2
config.set_decoder_custom_args(lut_config)

multi_config = qec.multi_decoder_config()
multi_config.decoders = [config]

with open("config.yaml", 'w') as f:
    f.write(multi_config.to_yaml_str(200))
print("Saved config to config.yaml")

With the decoder settings specified, all you need to do is load the configuration file with `configure_decoders_from_file()` and then use `cudaq.run()` specifying the `qec_circuit` you defined above. Recall that in the CUDA-Q kernel, you designated a decoder ID, so your main kernel could send syndromes to multiple decoders each with different settings depending on the needs.

This experiment is run on the `stim` backend, but it can easily retarget to a physical QPU to perform real-time error correction on a device!

In [ ]:
print("\nStep 2: Running circuit with decoding...")

qec.configure_decoders_from_file("config.yaml")

run_result = cudaq.run(qec_circuit, shots_count=10)

print("Ran 10 shots")

qec.finalize_decoders()

print("\nDone!")

A similar workflow was used to obtain the first ever real-time decoding results from a physical QPU. NVIDIA partnered with Quantinuum to use a relay-BP decoder to decode a 30-qubit qLDPC code on Quantinuum's Helios device. Experiments resulted in a median decode time of 67 microseconds and over a 5X error reduction from 4.95 to 0.925 thanks to decoding. This was also enabled by NVIDIA's low-latency NVQLink interconnect. You can read more about the work in the blog [here](https://developer.nvidia.com/blog/nvidia-nvqlink-architecture-integrates-accelerated-computing-with-quantum-processors/).

## Conclusion

As real-time error correction continues to mature, it is critically important to develop intuition for what such a procedure requires. You now have an understanding of DEMs and how they provide a robust means to capture a whole host of different errors rather than just errors within the encoded QEC memory. CUDA-Q QEC provides the infrastructure to prepare a decoder with a DEM (among other settings) and run real-time error correction through simple API calls within a kernel.

Everything discussed in this notebook complements previous lessons you have completed as it is agnostic of which specific QEC code you are using in a real-time experiment.

**Related Notebooks:**
* [QEC Decoders](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/04_QEC_Decoders.ipynb) — covers decoder fundamentals that this notebook builds upon
* [QEC Decoder Metrics and Parallel Window Decoding](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/08_QEC_Decoder_metrics_and_parallel_window_decoding.ipynb) — explores decoder throughput and reaction time metrics referenced in this notebook
* [QEC Stabilizers](https://github.com/NVIDIA/cuda-q-academic/blob/main/qec101/02_QEC_Stabilizers.ipynb) — introduces stabilizer formalism used to construct detectors